# Feature Engineering – Practical Machine Learning Series

## Learning objectives
By the end of this notebook, you should be able to explain the concept, identify appropriate techniques, implement them with pandas/scikit-learn, and recognize common risks.

**Important:** Feature engineering must be learned from the training data only when a transformation learns statistics or target relationships. Fit transformations on training data and apply the fitted transformation to validation/test data.

# 04 – Date & Time Feature Engineering

Dates and timestamps contain useful calendar and behavioral information. First parse them correctly, define the relevant timezone, and confirm the event timestamp is available at prediction time.

In [ ]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "birth_date": ["1995-04-12", "1988-11-20", "2001-02-05", "1992-07-30"],
    "join_date": ["2022-01-10", "2020-06-15", "2024-03-01", "2021-09-22"],
    "last_purchase": ["2026-10-20", "2026-09-01", "2026-10-30", "2026-08-15"],
    "purchase_date": ["2026-11-01", "2026-10-31", "2026-11-07", "2026-11-08"]
})
for col in ["birth_date", "join_date", "last_purchase", "purchase_date"]:
    data[col] = pd.to_datetime(data[col], errors="coerce")
as_of_date = pd.Timestamp("2026-11-10")
display(data)

## 1. Calendar components

From a timestamp, you can derive:
- year, month, day;
- day of week (Monday=0 through Sunday=6);
- ISO week number;
- quarter;
- weekend indicator;
- month-start and month-end indicators.

Use these when calendar patterns may affect the target, such as sales seasonality, staffing demand, or weekend purchasing. Avoid treating month numbers as a smooth numeric scale if the pattern is cyclical; sine/cosine features can represent cycles.

In [ ]:
dt = data["purchase_date"]
data["purchase_year"] = dt.dt.year
data["purchase_month"] = dt.dt.month
data["purchase_day"] = dt.dt.day
data["purchase_day_of_week"] = dt.dt.dayofweek
data["purchase_week_number"] = dt.dt.isocalendar().week.astype("Int64")
data["purchase_quarter"] = dt.dt.quarter
data["is_weekend_purchase"] = (dt.dt.dayofweek >= 5).astype("int")
data["is_month_start"] = dt.dt.is_month_start.astype("int")
data["is_month_end"] = dt.dt.is_month_end.astype("int")

# Cyclical representation: nearby months remain close around year-end.
data["month_sin"] = np.sin(2 * np.pi * data["purchase_month"] / 12)
data["month_cos"] = np.cos(2 * np.pi * data["purchase_month"] / 12)
display(data[["purchase_date", "purchase_month", "purchase_day_of_week",
              "purchase_week_number", "purchase_quarter", "is_weekend_purchase",
              "is_month_start", "is_month_end", "month_sin", "month_cos"]])

## 2. Age, tenure, recency, and time differences

- **Customer age:** elapsed time from birth date to a defined as-of date.
- **Employee tenure:** elapsed time from joining date to a defined as-of date.
- **Recency:** time since the last relevant event (e.g. last purchase).
- **Time difference:** elapsed time between two timestamps.
- **Days between transactions:** difference between consecutive transaction dates for the same customer.

Use a fixed prediction/as-of timestamp for historical training examples. Using today's date for every historical row can create inconsistent or future-aware features.

In [ ]:
data["customer_age_years"] = (
    (as_of_date - data["birth_date"]).dt.days / 365.2425
).round(1)
data["employee_tenure_days"] = (as_of_date - data["join_date"]).dt.days
data["employee_tenure_years"] = (data["employee_tenure_days"] / 365.2425).round(2)
data["days_since_last_purchase"] = (as_of_date - data["last_purchase"]).dt.days
data["days_from_purchase_to_asof"] = (as_of_date - data["purchase_date"]).dt.days
display(data[["customer_id", "customer_age_years", "employee_tenure_days",
              "employee_tenure_years", "days_since_last_purchase"]])

## 3. Frequency and days between transactions

Frequency is a count of events in a defined time window, such as purchases in the last 30 days. The window must end at the prediction timestamp. Days between transactions measures the gap between successive events.

For multiple transactions per customer, sort by customer and timestamp before using `groupby().diff()`.

In [ ]:
transactions = pd.DataFrame({
    "customer_id": [1, 1, 1, 2, 2, 3],
    "transaction_date": pd.to_datetime([
        "2026-10-01", "2026-10-08", "2026-10-25",
        "2026-09-10", "2026-10-10", "2026-11-02"
    ]),
    "amount": [100, 150, 80, 300, 120, 500]
}).sort_values(["customer_id", "transaction_date"])

transactions["days_since_previous_transaction"] = (
    transactions.groupby("customer_id")["transaction_date"].diff().dt.days
)
display(transactions)

# Example frequency feature: number of transactions per customer in this sample.
frequency = transactions.groupby("customer_id").size().rename("transaction_frequency")
display(frequency)

## 4. Seasonality

Seasonality is a repeating pattern over calendar periods (day, week, month, quarter, year). Examples include higher retail demand during holidays or different claim volumes by season.

Useful features:
- month/quarter/day-of-week;
- weekend and holiday indicators;
- cyclical sine/cosine encoding;
- lag and rolling-window features for time series.

For time-series prediction, use chronological splits. Rolling or lag features must be shifted so they contain only past information.

In [ ]:
# Illustrative lag and rolling features for a single time series.
daily = pd.DataFrame({
    "date": pd.date_range("2026-10-01", periods=8, freq="D"),
    "sales": [10, 12, 9, 15, 18, 14, 20, 22]
}).sort_values("date")

daily["sales_lag_1"] = daily["sales"].shift(1)
daily["sales_rolling_mean_3"] = daily["sales"].shift(1).rolling(3).mean()
display(daily)